# Supervised Fine-Tuning with SFTTrainer

This notebook demonstrates how to fine-tune the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer` from the `trl` library. The notebook cells run and will finetune the model. You can select your difficulty by trying out different datasets.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Fine-Tuning SmolLM2 with SFTTrainer</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p> 
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the `HuggingFaceTB/smoltalk` dataset</p>
    <p>🐕 Try out the `bigcode/the-stack-smol` dataset and finetune a code generation model on a specific subset `data/python`.</p>
    <p>🦁 Select a dataset that relates to a real world use case your interested in</p>
</div>

In [1]:
# Install the requirements in Google Colab
# !pip install transformers datasets trl huggingface_hub

import os
from huggingface_hub import login
from dotenv import load_dotenv

# Authenticate to Hugging Face
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
from google.api_core import retry
from google.generativeai.types import RequestOptions
import re
from typing import Dict
from typing import List
import tqdm
import google.generativeai as genai


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Configure the API key for gemini GenerativeAI
genai.configure(api_key=os.getenv("GEMINI_TOKEN"))

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset"
finetune_tags = ["smol-course", "module_1"]

# Generate with the base model

Here we will try out the base model which does not have a chat template. 

In [3]:
# # Let's test the base model before training
# prompt = "Write a haiku about programming"

# # Format with template
# messages = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# # Generate response
# inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
# outputs = model.generate(**inputs, max_new_tokens=100)
# print("Before training:")
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [4]:
prompt = "Write a haiku"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Before training:
user
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku
Write a haiku



In [5]:
# prompt = "Write about programming"

# # Format with template
# messages = [{"role": "user", "content": prompt}]
# formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# # Generate response
# inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
# outputs = model.generate(**inputs, max_new_tokens=100)
# print("Before training:")
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Dataset Preparation

We will load a sample dataset and format it for training. The dataset should be structured with input-output pairs, where each input is a prompt and the output is the expected response from the model.

**TRL will format input messages based on the model's chat templates.** They need to be represented as a list of dictionaries with the keys: `role` and `content`,.

In [6]:
# Load a sample dataset
from datasets import load_dataset

# Define dataset and config using the path and name parameters
ds_smoltalk = load_dataset(path="HuggingFaceTB/smoltalk", name="self-oss-instruct")
ds = load_dataset(path="bigcode/the-stack-smol", data_dir="data/python").shuffle(seed=42)

In [7]:
ds_smoltalk["train"][2]["messages"]

[{'content': 'Create a Python function to compute the sum of all the even elements in a list. The function should also print a message if the input list is empty. The function should also return the result as a list with an integer.',
  'role': 'user'},
 {'content': 'You can create a Python function to compute the sum of all the even elements in a list. If the input list is empty, the function should print a message and return an empty list. The function should also return the result as a list with an integer.\n\nHere\'s how you could do it:\n\n```python\ndef sum_even_elements(elements):\n    if not elements:\n        print("The input list is empty.")\n        return []\n\n    even_sum = sum(element for element in elements if element % 2 == 0)\n\n    return [even_sum]\n```\n\nThis function checks if the input list is empty and prints a message if it is. If the list is not empty, it computes the sum of the even elements using a list comprehension and returns the result as a list with an

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['content', 'avg_line_length', 'max_line_length', 'alphanum_fraction', 'licenses', 'repository_name', 'path', 'size', 'lang'],
        num_rows: 10000
    })
})

In [9]:
ds["train"].to_pandas()[["size"]].describe()

,size
count,10000.000000
mean,8087.089600
std,25177.031101
min,7.000000
25%,986.000000
50%,2760.500000
75%,7288.250000
max,927988.000000


In [10]:
ds_subset = ds["train"].filter(lambda x: x["size"] < 500 and x["max_line_length"] < 82)
ds_subset = ds_subset#.select(range(10))
ds_subset

In [11]:
ds_subset.select(range(5)).to_pandas()

,content,avg_line_length,max_line_length,alphanum_fraction,licenses,repository_name,path,size,lang
0,from django.conf.urls.defaults import *\n\nurl...,15.333333,47,0.750000,[BSD-3-Clause],SEL-Columbia/commcare-hq,corehq/apps/ivr/urls.py,92,Python
1,from django.contrib import admin\nfrom .models...,22.500000,39,0.829630,[MIT],functioncall/rescue-habit,blog/admin.py,135,Python
2,#!/usr/bin/env python3\ndef sum_recursin(numLi...,21.818182,53,0.629167,[BSD-2-Clause],zzz0072/Python_Exercises,07_RSI/ch03/sum.py,240,Python
3,from sklearn2sql_heroku.tests.regression impor...,28.200000,71,0.815603,[BSD-3-Clause],antoinecarme/sklearn2sql_heroku,tests/regression/diabetes/ws_diabetes_Gradient...,141,Python
4,from golem import actions\n\n\ndescription = '...,26.615385,73,0.719653,[MIT],kangchenwei/keyautotest2,projects/golem_integration/tests/actions/verif...,346,Python


### Helper methods

In [12]:
def generate_prompts_from_code_examples(code_examples: List[str]) -> List[str]:
    """
    Generate a list of prompts by inserting multiple Python code examples into the
    new prompt format specified below.

    Parameters
    ----------
    code_examples : List[str]
        A list of Python code snippets as strings to be analyzed and inserted into
        the prompt template.

    Returns
    -------
    List[str]
        A list of formatted prompts with each code example inserted into the template.
    """
    new_prompt_template = (
        "<python_code>\n"
        "{{PYTHON_CODE}}\n"
        "</python_code>\n\n"
        "Please follow these steps:\n\n"
        "1. Carefully read and analyze the provided Python code snippet.\n"
        "2. Break down the code inside <code_breakdown> tags:\n"
        "   - List and number each main component (functions, classes, algorithms)\n"
        "   - For each component, briefly describe its purpose and key features\n"
        "   - Note any important libraries or modules imported\n"
        "   - Highlight any unique or complex coding patterns\n\n"
        "3. Based on your analysis, create a brief text prompt that would instruct an AI to generate similar code. "
        "The prompt should be concise yet informative.\n\n"
        "Present your response in the following format:\n\n"
        "<analysis>\n"
        "[Your brief analysis of the code, highlighting only the most crucial points. Keep this section concise.]\n"
        "</analysis>\n\n"
        "<prompt>\n"
        "[Your concise text prompt for generating similar code. Focus on brevity while maintaining clarity.]\n"
        "</prompt>"
    )

    prompts = []
    for code_snippet in code_examples:
        formatted_prompt = new_prompt_template.replace("{{PYTHON_CODE}}", code_snippet)
        prompts.append(formatted_prompt)
    return prompts


def parse_prompt_from_completion(completion: str) -> Dict[str, str]:
    """
    Parse the <analysis> and <prompt> sections from a given completion string.
    If the <prompt> section is not properly enclosed (i.e., missing </prompt>),
    gather all text after <prompt>.

    Parameters
    ----------
    completion : str
        The completion string containing <analysis> and possibly <prompt> sections.

    Returns
    -------
    Dict[str, str]
        A dictionary with two keys, 'analysis' and 'prompt', each containing the
        corresponding text extracted from the completion.
    
    Examples
    --------
    >>> completion_text = \"\"\"<analysis>
    Analysis goes here
    </analysis>
    Some more text here
    <prompt>
    This is a test prompt without a closing tag.
    \"\"\"
    >>> result = parse_prompt_from_completion(completion_text)
    >>> result["analysis"]
    'Analysis goes here'
    >>> result["prompt"]
    'This is a test prompt without a closing tag.'
    """

    # Regex patterns to capture the text within <analysis>...</analysis>.
    pattern_analysis = r"<analysis>\s*(.*?)\s*</analysis>"
    analysis_match = re.search(pattern_analysis, completion, re.DOTALL)
    analysis = analysis_match.group(1).strip() if analysis_match else ""

    # Attempt to find a properly closed <prompt>...</prompt>.
    pattern_prompt_full = r"<prompt>\s*(.*?)\s*</prompt>"
    prompt_match_full = re.search(pattern_prompt_full, completion, re.DOTALL)

    if prompt_match_full:
        # If a closed <prompt> tag is found, use it.
        prompt = prompt_match_full.group(1).strip()
    else:
        # If we didn't find a closed <prompt>...</prompt>, look for <prompt>
        # and gather all text after it.
        prompt_start_match = re.search(r"<prompt>\s*(.*)$", completion, re.DOTALL)
        if prompt_start_match:
            prompt = prompt_start_match.group(1).strip()
        else:
            prompt = ""

    return {
        "analysis": analysis,
        "prompt": prompt
    }

def process_dataset(sample):
    # Convert the sample into a chat format

    messages = [
        {"role": "user", "content": sample["question"]},
        {"role": "assistant", "content": sample["answer"]}
        ]
    sample = {
        "messages": 
            messages
            #tokenizer.apply_chat_template(messages, tokenize=True)
        
        
    }
    return sample


retry_policy = retry.Retry(
    initial=10,  # First retry after 10 seconds
    multiplier=2,  # Double the wait time for each retry
    maximum=75,  # Max wait time between retries
    timeout=300  # Total retry time limit (5 minutes)
)


def generate_with_retry(prompt):
    model = genai.GenerativeModel("gemini-1.5-flash")
    response = model.generate_content(
        prompt,
        request_options=RequestOptions(retry=retry_policy)
    )
    return response

### Generate prompts from the code examples in `bigcode/the-stack-smol`

In [13]:
content = ds_subset["content"]
prompts = generate_prompts_from_code_examples(content)
messages = [{"role": "user", "content": p} for p in prompts]
# from huggingface_hub import InferenceClient
# client = InferenceClient("meta-llama/Meta-Llama-3-8B-Instruct") Results from this smaller model are not as good
# completions = [client.chat_completion([message], max_tokens=6000) for message in messages]
completions = []
# progress bar for the completions generation process

for tick, p in enumerate(tqdm.tqdm(prompts)):
    completion = generate_with_retry(p)
    if tick % 100 == 0:
        print(p)
        print("----")
        print(completion.text)
        print("----")
    completions.append(completion)

  0%|          | 0/1189 [00:00<?, ?it/s]

  0%|          | 1/1189 [00:02<55:04,  2.78s/it]

<python_code>
from django.conf.urls.defaults import *

urlpatterns = patterns("corehq.apps.ivr.views",
)


</python_code>

Please follow these steps:

1. Carefully read and analyze the provided Python code snippet.
2. Break down the code inside <code_breakdown> tags:
   - List and number each main component (functions, classes, algorithms)
   - For each component, briefly describe its purpose and key features
   - Note any important libraries or modules imported
   - Highlight any unique or complex coding patterns

3. Based on your analysis, create a brief text prompt that would instruct an AI to generate similar code. The prompt should be concise yet informative.

Present your response in the following format:

<analysis>
[Your brief analysis of the code, highlighting only the most crucial points. Keep this section concise.]
</analysis>

<prompt>
[Your concise text prompt for generating similar code. Focus on brevity while maintaining clarity.]
</prompt>
----
<analysis>
This code snip

  8%|▊         | 101/1189 [05:31<27:08,  1.50s/it] 

<python_code>
#!/usr/bin/python3
import glob
from PIL import Image

# get all the jpg files from the current folder
for infile in glob.glob("*.jpg"):
  im = Image.open(infile)
  # convert to thumbnail image
  im.thumbnail((500, 500), Image.ANTIALIAS)
  # don't save if thumbnail already exists
  if infile[0:2] != "T_":
    # prefix thumbnail file with T_
    im.save("thumbs/T_" + infile, "JPEG")

</python_code>

Please follow these steps:

1. Carefully read and analyze the provided Python code snippet.
2. Break down the code inside <code_breakdown> tags:
   - List and number each main component (functions, classes, algorithms)
   - For each component, briefly describe its purpose and key features
   - Note any important libraries or modules imported
   - Highlight any unique or complex coding patterns

3. Based on your analysis, create a brief text prompt that would instruct an AI to generate similar code. The prompt should be concise yet informative.

Present your response in the follo

...

 84%|████████▍ | 1001/1189 [1:00:40<05:59,  1.91s/it]

<python_code>
with open("input") as file:
    massList = file.readlines()

def calcModuleFuel(fuel, mass):
    addedFuel = int(mass/3)-2
    if addedFuel <=0:
        return fuel
    else:
        return calcModuleFuel(fuel+addedFuel, addedFuel)

def calcFuelSum(massList):
    fuelSum = 0
    for m in massList:
        m = float(m)
        fuelSum  += calcModuleFuel(0, m)
    return fuelSum

print calcFuelSum(massList)
</python_code>

Please follow these steps:

1. Carefully read and analyze the provided Python code snippet.
2. Break down the code inside <code_breakdown> tags:
   - List and number each main component (functions, classes, algorithms)
   - For each component, briefly describe its purpose and key features
   - Note any important libraries or modules imported
   - Highlight any unique or complex coding patterns

3. Based on your analysis, create a brief text prompt that would instruct an AI to generate similar code. The prompt should be concise yet informative.

Present yo

 93%|█████████▎| 1101/1189 [1:06:31<04:46,  3.26s/it]

<python_code>
from pyowm import OWM
owm = OWM('21ff51d901692fd3e2f5ecc04d3617f1')
place = input('Input Place: ')
mgr = owm.weather_manager()
observation = mgr.weather_at_place(place)
w = observation.weather
wind = w.detailed_status
t = w.temperature('celsius')
print(wind)
print(t)
exit_ = input('')

</python_code>

Please follow these steps:

1. Carefully read and analyze the provided Python code snippet.
2. Break down the code inside <code_breakdown> tags:
   - List and number each main component (functions, classes, algorithms)
   - For each component, briefly describe its purpose and key features
   - Note any important libraries or modules imported
   - Highlight any unique or complex coding patterns

3. Based on your analysis, create a brief text prompt that would instruct an AI to generate similar code. The prompt should be concise yet informative.

Present your response in the following format:

<analysis>
[Your brief analysis of the code, highlighting only the most crucial poin

100%|██████████| 1189/1189 [1:11:10<00:00,  3.59s/it]


In [14]:
len(completions)

1189

### Format into train and test dataset and save to file

In [15]:
contents = [parse_prompt_from_completion(c.text) for c in completions]
questions = [c["prompt"] for c in contents]
dataset = ds_subset.add_column("question", questions)
dataset = dataset.rename_column("content", "answer")
dataset = dataset.map(process_dataset, remove_columns=dataset.features)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

# add timestamp to filename
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
dataset["train"].to_json(f"sft_data/train_dataset_{timestamp}.json", orient="records")
dataset["test"].to_json(f"sft_data/test_dataset_{timestamp}.json", orient="records")
dataset["train"][2]["messages"]

Map:   0%|          | 0/1189 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

[{'content': 'Generate Python code that defines configuration parameters for a chat application.  The parameters should include database path (relative to the project root), port range, a debug flag (obtained from the CHAT_APP_DEBUG environment variable), and a timeout value (set to 30 if debug is True, otherwise 0.5).  Use pathlib for path handling.',
  'role': 'user'},
 {'content': 'import os\nfrom pathlib import Path\n\nDB_NAME = "chatapp.db"\nPROJECT_PATH = Path(__file__).parents[1]\nDB_PATH = os.path.join(PROJECT_PATH, "resource", DB_NAME)\n\nPORT_MIN = 1024\nPORT_MAX = 65535\n\nDEBUG = os.getenv("CHAT_APP_DEBUG", False)\n\nif DEBUG:\n    TIMEOUT = 30\nelse:\n    TIMEOUT = 0.5\n',
  'role': 'assistant'}]

### Load dataset from disk

In [16]:
dataset = load_dataset(
    "json",
    data_files={
        "train": "sft_data/train_*.json",
        "test": "sft_data/test_*.json",
    },
)
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1070
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 119
    })
})

## Configuring the SFTTrainer

The `SFTTrainer` is configured with various parameters that control the training process. These include the number of training steps, batch size, learning rate, and evaluation strategy. Adjust these parameters based on your specific requirements and computational resources.

In [17]:
# # Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    evaluation_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    tokenizer=tokenizer,
    eval_dataset=dataset["test"],
)

# TODO: 🦁 🐕 align the SFTTrainer params with your chosen dataset. For example, if you are using the `bigcode/the-stack-smol` dataset, you will need to choose the `content` column`

## Training the Model

With the trainer configured, we can now proceed to train the model. The training process will involve iterating over the dataset, computing the loss, and updating the model's parameters to minimize this loss.

In [18]:
# Train the model
trainer.train()

# Save the model
trainer.save_model(f"./{finetune_name}")

NameError: name 'trainer' is not defined

In [ ]:
trainer.push_to_hub(tags=finetune_tags)

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Generate with fine-tuned model</h2>
    <p>🐕 Use the fine-tuned to model generate a response, just like with the base example..</p>
</div>

In [ ]:
# Test the fine-tuned model on the same prompt

# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=10000)
# Model is yet not strong enough to know about Haiku.
# The model does now output code.
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Output from model:
```
user
Write a haiku about programming
assistant
# -*- coding: utf-8 -*-
# @Time    : 2019/10/25 15:25
# @Author  : (removed)
# @User    : (removed)
# @File    : 20191025_1633.py
# @Software: PyCharm

from . import haiku

haiku.add("Programming", "Programming is a fun way to learn programming.")
haiku.add("Programming", "Programming is a fun way to learn programming.")
...
```

## 💐 You're done!

This notebook provided a step-by-step guide to fine-tuning the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer`. By following these steps, you can adapt the model to perform specific tasks more effectively. If you want to carry on working on this course, here are steps you could try out:

- Try this notebook on a harder difficulty
- Review a colleagues PR
- Improve the course material via an Issue or PR.